# 02 — Phase 1 Scalability and Overhead

Curated starter notebook for aggregation overhead analysis.

This notebook reads local scaling fixtures in `results/expected/` and visualizes round-time and aggregation-time costs across defenses.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EXPECTED_DIR = ROOT / "results" / "expected"
assert EXPECTED_DIR.exists(), f"Missing expected fixtures at {EXPECTED_DIR}"


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

In [ ]:
scaling_files = sorted(EXPECTED_DIR.glob("akash_scaling_*_3c.json"))
assert scaling_files, "No scaling fixtures found"

rows = []
for path in scaling_files:
    payload = load_json(path)
    defense = payload["config"]["defense"]
    final = payload["rounds"][-1]
    rows.append({
        "defense": defense,
        "round_time": float(final["round_time"]),
        "agg_time": float(final["agg_time"]),
        "perplexity": float(final["perplexity"]),
    })

rows = sorted(rows, key=lambda x: x["agg_time"])
rows

In [ ]:
# Plot aggregation and round-time overhead by defense.
defenses = [row["defense"] for row in rows]
agg = [row["agg_time"] for row in rows]
round_t = [row["round_time"] for row in rows]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(defenses, agg, color="#72B7B2")
axes[0].set_title("Scaling Attack: Aggregation Time")
axes[0].set_ylabel("seconds")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(defenses, round_t, color="#F58518")
axes[1].set_title("Scaling Attack: Round Time")
axes[1].set_ylabel("seconds")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Defense Overhead Profile (Expected Fixtures)", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Relative aggregation-time overhead vs FedAvg.
fedavg_agg = next(row["agg_time"] for row in rows if row["defense"] == "FedAvg")
overhead = [{"defense": row["defense"], "agg_overhead_x": row["agg_time"] / fedavg_agg} for row in rows]
overhead

In [ ]:
# Visualize overhead multipliers.
labels = [row["defense"] for row in overhead]
values = [row["agg_overhead_x"] for row in overhead]

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color="#54A24B")
plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title("Aggregation Overhead vs FedAvg")
plt.ylabel("x over FedAvg")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion

- Aggregation cost differs substantially by defense, even in small-client fixtures.
- This notebook is the staging ground for compute-cost figures in the paper's scalability section.